Load data

In [35]:
# get text from sample pdfs
from src.basic_code_search.document_loader import load_pdf_documents

text = load_pdf_documents(folder_path="data")
print(f"Loaded {len(text)} characters from PDF files.")


Loaded 128973 characters from PDF files.


In [36]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_text(text)

print(f"Split into {len(chunks)} chunks.")

Split into 289 chunks.


In [37]:
from src.basic_code_search.embedding_model import EmbeddingModel

embedding_model = EmbeddingModel()
embeddings = embedding_model.encode(chunks)

print(f"Generated {len(embeddings)} embeddings.")

Generated 289 embeddings.


In [38]:
from src.basic_code_search.search_engine import SearchEngine

search_engine = SearchEngine(embedding_model=embedding_model, collection_name="basic_search_db", top_k=3)
search_engine.open()
search_engine.load_text_data(chunks)
query = "What is a transformer model?"
results = search_engine.search(query)
for idx, result in enumerate(results):
    print(f"\nResult {result.id} (Score: {result.score}):\n{result.payload['text']}")
search_engine.close()


Result 338a074c-2691-45e8-a0cf-06cd65436312 (Score: 0.5118749):
language modeling tasks [34].
To the best of our knowledge, however, the Transformer is the first transduction model relying
entirely on self-attention to compute representations of its input and output without using sequence-
aligned RNNs or convolution. In the following sections, we will describe the Transformer, motivate
self-attention and discuss its advantages over models such as [17, 18] and [9].
3 Model Architecture

Result 49a4de67-ff98-4f39-b6a6-81922c624549 (Score: 0.47969738):
block, computing hidden representations in parallel for all input and output positions. In these models,
the number of operations required to relate signals from two arbitrary input or output positions grows
in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes
it more difficult to learn dependencies between distant positions [ 12]. In the Transformer this is

Result cf95b223-c7c6-4b7a-8ebc-7e

# Task 2

In [ ]:
import mteb

# start search engine
search_engine = SearchEngine(embedding_model=embedding_model, collection_name="basic_search_db", top_k=3)
search_engine.open()

# get CosQA
task = mteb.get_tasks(tasks=["CosQA"])

# evaluate on CosQA
results = mteb.evaluate(
    model=search_engine.embedding_model.get_model(),
    tasks=task,
)[0]

print("Recall@10: ")
for result in results:
    tests = result.scores['test']
    for test in tests:
        print(test['recall_at_10'])
        print(test['mrr_at_10'])
        print(test['ndcg_at_10'])

# close search engine
search_engine.close()

Evaluating tasks:   0%|          | 0/1 [00:00<?, ?it/s]

Results for 0.32704:
0.562
0.241213
0.32704


In [51]:
from datasets import load_dataset

corpus = load_dataset("CoIR-Retrieval/cosqa", "corpus")
queries = load_dataset("CoIR-Retrieval/cosqa", "queries")
default = load_dataset("CoIR-Retrieval/cosqa", "default")

In [57]:
print(corpus['corpus']['_id'])

Column(['d1', 'd2', 'd3', 'd4', 'd5'])


In [ ]:
search_engine = SearchEngine(embedding_model=embedding_model, collection_name="basic_search_db", top_k=10)
search_engine.open()
search_engine.load_text_data(corpus['train']['text'], ids=corpus['train']['_id'])



Dataset({
    features: ['query-id', 'corpus-id', 'score'],
    num_rows: 19604
})


In [ ]:
search_engine.close()